## 1. Setup and Installation

Install required packages and configure your environment.

In [ ]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install -q python-dotenv google-adk

print("✅ Dependencies installed")

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"✅ Project root: {project_root}")

## 2. Configure Google Gemini API

**Required:** Get your free API key at https://aistudio.google.com/app/apikey

In [ ]:
# Option 1: Load from .env file (recommended)
load_dotenv()

# Option 2: Set directly (not recommended - don't commit this!)
# os.environ['GOOGLE_API_KEY'] = 'your-key-here'

# Verify API key
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print("✅ Google Gemini API configured")
    print(f"   Key: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("Please set your API key:")
    print("  1. Get free key: https://aistudio.google.com/app/apikey")
    print("  2. Create a .env file with: GOOGLE_API_KEY=your-key-here")
    print("     OR set environment: export GOOGLE_API_KEY='your-key-here'")

## 3. Define Your ODD Specification

Describe your robot's operational constraints in **natural language**. The AI agents will convert this to a formal specification and use it for compliance analysis.

**Customize this for your robot:**
- Indoor vs outdoor
- Terrain types
- Speed limits
- Lighting requirements
- Obstacle density
- Stability constraints

In [ ]:
# Natural Language ODD Description
# The AI will parse this and create a formal specification

odd_description = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Environment Type:
   - Designed for: indoor_office, indoor_corridor
   - Prohibited: outdoor environments, staircases, unstructured terrain

2. Lighting Conditions:
   - Required: bright or dim lighting (adequate visibility)
   - Prohibited: dark environments (requires vision sensors)

3. Terrain Type:
   - Designed for: smooth_floor (tile, hardwood, low-pile carpet)
   - Prohibited: rough or very_rough terrain, stairs, slopes >10°

4. Speed Range:
   - Normal operation: 0.0 to 1.5 m/s
   - Physical limit: 2.5 m/s (emergency only)

5. Obstacle Density:
   - Acceptable: low to moderate obstacles (0.0 to 0.6 normalized)
   - Boundary: 0.6 to 0.8 (crowded but navigable)
   - Prohibited: >0.8 (too cluttered for safe navigation)

6. Traversability:
   - Required: navigable space (0.5 to 1.0 score)
   - Boundary: 0.3 to 0.5 (challenging but possible)
   - Prohibited: <0.3 (impassable or unsafe)

7. Collision Risk:
   - Acceptable: low risk (0.0 to 0.3 likelihood)
   - Boundary: 0.3 to 0.5 (caution required)
   - Prohibited: >0.5 (high risk, stop immediately)

8. Platform Stability:
   - Required: stable platform (roll/pitch <15°)
   - Boundary: 15° to 20° (unstable but recoverable)
   - Prohibited: >20° (tip-over risk)
"""

print(f"✅ ODD specification defined ({len(odd_description)} characters)")
print("\n💡 TIP: Customize this for different robot types:")
print("   • Outdoor delivery robots (weather, GPS, terrain)")
print("   • Aerial drones (altitude, wind speed, battery)")
print("   • Warehouse AMRs (floor type, shelf proximity)")
print("   • Autonomous vehicles (road type, traffic, visibility)")

## 4. Select Scenario to Analyze

Choose which preprocessed dataset you want to analyze. Scenarios are stored in `data/processed/runs/`.

**Available scenarios:**
- `sim_run_test` - Small test dataset (2 windows)
- `demo_run` - Demo dataset for quick testing
- Your own scenarios (after running `extract_windows.py`)

In [ ]:
# List available scenarios
data_dir = project_root / "data" / "processed" / "runs"

if data_dir.exists():
    scenarios = [d.name for d in data_dir.iterdir() if d.is_dir()]
    print("📁 Available scenarios:")
    for i, scenario in enumerate(scenarios, 1):
        scenario_path = data_dir / scenario
        window_count = len(list(scenario_path.glob("motion_*.json")))
        print(f"   {i}. {scenario:20s} ({window_count:2d} windows)")
else:
    print("⚠️ Data directory not found!")
    print(f"   Expected: {data_dir}")
    print("   Run: python scripts/extract_windows.py first")
    scenarios = []

print()

# Select scenario (change this!)
SCENARIO_NAME = "sim_run_test"

if scenarios and SCENARIO_NAME in scenarios:
    scenario_path = data_dir / SCENARIO_NAME
    window_count = len(list(scenario_path.glob("motion_*.json")))
    print(f"✅ Selected: {SCENARIO_NAME}")
    print(f"   Windows: {window_count}")
    print(f"   Path: {scenario_path}")
else:
    print(f"❌ Scenario '{SCENARIO_NAME}' not found!")
    if scenarios:
        print(f"   Available: {', '.join(scenarios)}")

## 5. Run ODD Analysis Workflow

Execute the **10-agent sequential pipeline** that analyzes your scenario:

**Pipeline stages:**
1. **ODD Spec Agent** - Parse natural language ODD → formal specification
2. **Perception Agents** - Analyze camera + LiDAR images (loop + summary)
3. **Motion Agents** - Detect motion from IMU data (loop + summary)
4. **Collision Agents** - Assess collision risk (loop + summary)
5. **COD Classifier** - Determine current operating domain
6. **ODD Compliance** - Compare COD vs ODD, detect violations
7. **Report Generator** - Create comprehensive analysis report

**Note:** This uses the `odd_agents` module - the same code used by production scripts and tests.

In [ ]:
import json
from odd_agents import run_odd_workflow

print("🚀 Starting ODD analysis workflow...")
print(f"   Scenario: {SCENARIO_NAME}")
print(f"   ODD description: {len(odd_description)} characters")
print()
print("⏳ This may take 2-3 minutes...")
print()

try:
    # Run the workflow
    result = await run_odd_workflow(
        scenario_name=SCENARIO_NAME,
        nl_odd_description=odd_description
    )
    
    if result:
        print()
        print("=" * 80)
        print("✅ ANALYSIS COMPLETE!")
        print("=" * 80)
        
        # Quick summary
        report = result.get('report', {})
        metadata = report.get('scenario_metadata', {})
        compliance = result.get('full_analysis', {}).get('odd_compliance', {})
        
        print(f"\n📊 Summary:")
        print(f"   • Windows analyzed: {metadata.get('total_windows_analyzed', 'N/A')}")
        print(f"   • Data source: {metadata.get('data_source', 'N/A')}")
        print(f"   • ODD compliance: {compliance.get('overall_compliance', 'N/A')}")
        print(f"   • Violations: {len(compliance.get('violations', []))}")
        
    else:
        print()
        print("❌ Workflow failed - no results generated")
        result = None
        
except Exception as e:
    print()
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()
    result = None

## 6. View Executive Summary

High-level findings and key insights from the analysis.

In [ ]:
if result:
    report = result['report']
    
    print("=" * 80)
    print("EXECUTIVE SUMMARY")
    print("=" * 80)
    print()
    print(report.get('executive_summary', 'N/A'))
    print()
    
    print("=" * 80)
    print("KEY FINDINGS")
    print("=" * 80)
    for i, finding in enumerate(report.get('key_findings', []), 1):
        print(f"\n{i}. {finding}")
    
    print()
    print("=" * 80)
    print("RECOMMENDATIONS")
    print("=" * 80)
    for i, rec in enumerate(report.get('recommendations', []), 1):
        print(f"\n{i}. {rec}")
else:
    print("⚠️ No results available. Please run the workflow first (cell 5).")

## 7. ODD Compliance Details

Detailed breakdown of compliance status across all analyzed dimensions.

In [ ]:
if result:
    compliance = result['full_analysis']['odd_compliance']
    
    print("=" * 80)
    print("ODD COMPLIANCE ANALYSIS")
    print("=" * 80)
    print()
    print(f"Overall Status: {compliance.get('overall_compliance', 'N/A')}")
    print(f"Total Violations: {len(compliance.get('violations', []))}")
    print()
    
    if compliance.get('violations'):
        print("❌ VIOLATIONS DETECTED:")
        print("-" * 80)
        for violation in compliance['violations']:
            print(f"  • {violation}")
        print()
    else:
        print("✅ NO VIOLATIONS DETECTED")
        print()
    
    print("📊 CATEGORICAL COMPLIANCE:")
    print("-" * 80)
    for axis, status in compliance.get('categorical_compliance', {}).items():
        icon = "✅" if status == "IN_ODD" else ("⚠️" if status == "ODD_BOUNDARY" else "❌")
        print(f"{icon} {axis:30s} → {status}")
    
    print()
    print("📏 NUMERIC COMPLIANCE:")
    print("-" * 80)
    for axis, status in compliance.get('numeric_compliance', {}).items():
        icon = "✅" if status == "IN_ODD" else ("⚠️" if status == "ODD_BOUNDARY" else "❌")
        print(f"{icon} {axis:30s} → {status}")
else:
    print("⚠️ No results available.")

## 8. Visualize Results

Plot collision risk and motion detection over time.

In [ ]:
if result:
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Extract data
    collision_data = result['full_analysis']['collision']
    motion_data = result['full_analysis']['motion']
    
    per_window_collision = collision_data.get('per_window_collision', [])
    per_window_motion = motion_data.get('per_window_motion', [])
    
    if per_window_collision and per_window_motion:
        # Prepare data
        windows = [w['window_id'] for w in per_window_collision]
        risk_levels = [w['risk_level'] for w in per_window_collision]
        likelihoods = [w['likelihood'] for w in per_window_collision]
        motion_detected = [w['motion_detected'] for w in per_window_motion]
        motion_types = [w['motion_type'] for w in per_window_motion]
        
        # Create figure
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
        
        # Plot 1: Collision risk
        color_map = {'safe': 'green', 'caution': 'orange', 'alert': 'red'}
        colors = [color_map.get(level, 'gray') for level in risk_levels]
        
        ax1.bar(range(len(windows)), likelihoods, color=colors, alpha=0.7, edgecolor='black')
        ax1.axhline(y=0.3, color='orange', linestyle='--', linewidth=1, label='Boundary (0.3)')
        ax1.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='Alert (0.5)')
        ax1.set_ylabel('Collision Likelihood', fontsize=12, fontweight='bold')
        ax1.set_title('Collision Risk Assessment', fontsize=14, fontweight='bold')
        ax1.legend(loc='upper right')
        ax1.grid(axis='y', alpha=0.3)
        ax1.set_ylim(0, 1.0)
        
        # Plot 2: Motion detection
        motion_colors = ['red' if m else 'lightgray' for m in motion_detected]
        ax2.bar(range(len(windows)), motion_detected, color=motion_colors, alpha=0.7, edgecolor='black')
        ax2.set_ylabel('Motion Detected', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Window ID', fontsize=12, fontweight='bold')
        ax2.set_title('Motion Detection (IMU-based)', fontsize=14, fontweight='bold')
        ax2.set_xticks(range(len(windows)))
        ax2.set_xticklabels(windows)
        ax2.set_yticks([0, 1])
        ax2.set_yticklabels(['No Motion', 'Motion'])
        ax2.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        collision_stats = collision_data.get('overall_stats', {})
        motion_stats = motion_data.get('motion_stats', {})
        
        print()
        print("📊 STATISTICS:")
        print("-" * 80)
        print(f"Collision - Avg likelihood: {collision_stats.get('avg_likelihood', 0):.3f}")
        print(f"Collision - Alert level: {collision_stats.get('alert_count', 0)} windows")
        print(f"Motion - Detection rate: {motion_stats.get('motion_detection_rate', 0)*100:.1f}%")
        print(f"Motion - Max accel: {motion_stats.get('max_horizontal_accel_mps2', 0):.3f} m/s²")
    else:
        print("⚠️ No per-window data available for visualization")
else:
    print("⚠️ No results available.")

## 9. Export Results

Save the complete analysis report to JSON for further processing or sharing.

In [ ]:
if result:
    # Report is already saved by run_odd_workflow()
    output_path = data_dir / SCENARIO_NAME / "odd_analysis_report.json"
    
    if output_path.exists():
        file_size = output_path.stat().st_size / 1024
        print(f"✅ Results already saved to:")
        print(f"   {output_path}")
        print(f"   Size: {file_size:.1f} KB")
    else:
        print("⚠️ Report file not found")
    
    # You can also access the result directly
    print()
    print("💡 TIP: Access results programmatically:")
    print("   result['report']              # Human-readable report")
    print("   result['full_analysis']       # Complete sensor analysis")
    print("   result['odd_spec']            # Parsed ODD specification")
else:
    print("⚠️ No results to export.")

## Next Steps

### 🎯 Customize Your Analysis

1. **Modify the ODD** (cell 3) to match your robot's design constraints
2. **Test different scenarios** (cell 4) to analyze various datasets
3. **Adjust parameters** by editing the ODD description

### 📊 Advanced Usage

- **Compare scenarios**: Run the workflow on multiple datasets and compare results
- **Parameter sensitivity**: Test how ODD threshold changes affect compliance
- **Custom visualizations**: Access `result` dictionary to create your own plots

### 🔍 Dive Deeper

- **View agent implementations**: `odd_agents/agents/` directory
- **Understand the workflow**: `odd_agents/workflow.py`
- **Read documentation**: `docs/` directory
- **Factory pattern explanation**: `docs/FACTORY_PATTERN.md`

### 🚀 Production Deployment

To use this in production:
1. Preprocess ROS2 bags: `python scripts/extract_windows.py`
2. Run analysis: `python scripts/odd_workflow.py`
3. Automate with CI/CD for continuous monitoring

---

**Questions?** Check the project documentation or explore the source code!